# 2.1 — Architecture Definition
**AIKONIC — MobileNetV3-Small for Dysgraphia Detection**

This notebook:
- Builds the DysgraphiaCNN model (Lambda → MobileNetV3-Small → custom head)
- Verifies parameter count (~2.5M), output shape (None, 2)
- Confirms Grad-CAM target layer and Phase A freeze
- Saves architecture summary to `reports/architecture_summary.txt`

**Assigned to:** CNN Architect (JAY PAOLO MARCOS)

In [ ]:
import sys, os
# Go up one level from phase_02/ to reach the project root
sys.path.insert(0, os.path.abspath('..'))

import tensorflow as tf
import config

print(f'TensorFlow version : {tf.__version__}')
print(f'Project root       : {os.path.abspath("..")}')
print(f'Checkpoints dir    : {config.CHECKPOINTS_DIR}')
print(f'Reports dir        : {config.REPORTS_DIR}')

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Small
import io

def build_model(freeze_base=True, dropout_rate=config.DROPOUT_RATE,
                learning_rate=config.PHASE_A_LR):
    """
    DysgraphiaCNN: MobileNetV3-Small based binary classifier.
    Input : (224, 224, 1) grayscale
    Output: (None, 2)  →  [P(LPD), P(PD)]
    """
    inputs = tf.keras.Input(shape=(224, 224, 1), name='grayscale_input')

    # Lambda: 1-ch grayscale → 3-ch (bridges to ImageNet pretrained weights)
    x = layers.Lambda(
        lambda t: tf.repeat(t, 3, axis=-1),
        name='channel_replication'
    )(inputs)

    # MobileNetV3-Small base
    base = MobileNetV3Small(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3),
    )
    base.trainable = not freeze_base
    x = base(x, training=not freeze_base)

    # Classification head
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.BatchNormalization(name='batch_norm')(x)
    x = layers.Dense(config.DENSE_UNITS, activation=config.ACTIVATION, name='dense_128')(x)
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = layers.Dense(config.NUM_CLASSES, activation='softmax', name='classifier')(x)

    model = Model(inputs=inputs, outputs=outputs, name='DysgraphiaCNN')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

print('build_model() defined.')

In [ ]:
# ── Build Phase A model (frozen base) ─────────────────────────────────────────
print('Building DysgraphiaCNN (Phase A — frozen base)...')
model = build_model(freeze_base=True)
model.summary()

In [ ]:
# ── Save architecture summary ──────────────────────────────────────────────────
os.makedirs(config.REPORTS_DIR, exist_ok=True)
stream = io.StringIO()
model.summary(print_fn=lambda line: stream.write(line + '\n'))
with open(config.ARCH_SUMMARY, 'w') as f:
    f.write(stream.getvalue())
print(f'Architecture summary saved → {config.ARCH_SUMMARY}')

In [ ]:
# ── Verify parameter count ─────────────────────────────────────────────────────
total_params     = model.count_params()
trainable_params = int(sum(tf.size(w).numpy() for w in model.trainable_weights))
frozen_params    = int(sum(tf.size(w).numpy() for w in model.non_trainable_weights))

print(f'Total parameters     : {total_params:>10,}')
print(f'Trainable parameters : {trainable_params:>10,}  ← classification head only (Phase A)')
print(f'Frozen parameters    : {frozen_params:>10,}  ← MobileNetV3-Small base')

assert total_params > 2_000_000, f'Expected ~2.5M params, got {total_params:,}'
print('✓ Parameter count verified (>2M, consistent with MobileNetV3-Small)')

In [ ]:
# ── Verify output shape ────────────────────────────────────────────────────────
dummy = tf.zeros((1, 224, 224, 1))
out   = model(dummy, training=False)
print(f'Input  shape: {dummy.shape}')
print(f'Output shape: {out.shape}')
assert out.shape == (1, 2), f'Expected (1, 2), got {out.shape}'
print('✓ Output shape (None, 2) verified')

In [ ]:
# ── Verify Lambda channel-replication layer ────────────────────────────────────
layer_names = [l.name for l in model.layers]
assert 'channel_replication' in layer_names, 'Lambda layer not found!'
print('✓ Lambda (channel_replication) layer confirmed')

# ── Verify Grad-CAM target layer ───────────────────────────────────────────────
def find_layer(m, name):
    for l in m.layers:
        if l.name == name: return l
        if hasattr(l, 'layers'):
            found = find_layer(l, name)
            if found: return found
    return None

gradcam_layer = find_layer(model, config.GRAD_CAM_LAYER)
if gradcam_layer:
    print(f'✓ Grad-CAM layer "{config.GRAD_CAM_LAYER}" confirmed in model')
else:
    print(f'⚠  "{config.GRAD_CAM_LAYER}" not found at top level — check inside MobileNetV3Small sub-model')

In [ ]:
# ── Confirm Phase A: only classification head is trainable ─────────────────────
trainable_names = [w.name for w in model.trainable_weights]
head_keywords   = ['global_avg_pool', 'batch_norm', 'dense_128', 'dropout', 'classifier']
non_head = [n for n in trainable_names if not any(kw in n for kw in head_keywords)]

if len(non_head) == 0:
    print('✓ Only classification head is trainable — MobileNetV3-Small base is fully frozen')
else:
    print(f'⚠  {len(non_head)} non-head weights are trainable — check freeze logic')
    for n in non_head[:5]: print(f'   {n}')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────────
print('=' * 60)
print('  2.1 ARCHITECTURE DEFINITION COMPLETE')
print('=' * 60)
print(f'  Grad-CAM target layer : {config.GRAD_CAM_LAYER}')
print(f'  Phase A LR            : {config.PHASE_A_LR}')
print(f'  Phase B LR            : {config.PHASE_B_LR}')
print(f'  Architecture summary  : {config.ARCH_SUMMARY}')
print()